**Google Colab Implementation for Gemma-4-4B-it**

In [ ]:
!pip install -q -U bitsandbytes transformers git+https://github.com/huggingface/peft.git accelerate trl datasets "torchao>=0.16.0"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import torch
from transformers import Trainer, AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments, DataCollatorForLanguageModeling
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from datasets import load_dataset

**Load Model in 4-Bit (QLoRA)**

In [ ]:
model_id = "google/gemma-4-31b-it"
smaller_model_id = "google/gemma-4-E4B-it"

# 4-bit Quantization Config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# Load Tokenizer and Model
tokenizer = AutoTokenizer.from_pretrained(smaller_model_id)
model = AutoModelForCausalLM.from_pretrained(
    smaller_model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

# Prepare for k-bit training
model = prepare_model_for_kbit_training(model)
print("Model loaded and quantized!")

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/16.0G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

Model loaded and quantized!


**Load and Format the LR-Sum Khmer Dataset**

The lr-sum dataset has a specific structure. We need to map its text and summary columns into the Gemma instruction format.

In [ ]:
# Load the Khmer subset of lr-sum
dataset = load_dataset("bltlab/lr-sum", "khm")

def format_gemma_prompt(example):
    instruction = "Please provide a concise and professional summary of the following Khmer text."
    user_prompt = f"{instruction}\n\n{example['text']}"
    model_response = example['summary']

    # EXACT Gemma-4 Instruction Format
    full_text = f"<start_of_turn>user\n{user_prompt}<end_of_turn>\n<start_of_turn>model\n{model_response}<end_of_turn>"
    return {"text": full_text}

# Map the formatting function
train_dataset = dataset["train"].map(format_gemma_prompt, remove_columns=dataset["train"].column_names)
test_dataset = dataset["validation"].map(format_gemma_prompt, remove_columns=dataset["validation"].column_names)

print(f"Dataset ready. Training samples: {len(train_dataset)}")

README.md: 0.00B [00:00, ?B/s]

khm/test-00000-of-00001.parquet:   0%|          | 0.00/2.13M [00:00<?, ?B/s]

khm/train-00000-of-00001.parquet:   0%|          | 0.00/16.9M [00:00<?, ?B/s]

khm/validation-00000-of-00001.parquet:   0%|          | 0.00/2.07M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/486 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/3888 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/486 [00:00<?, ? examples/s]

Map:   0%|          | 0/3888 [00:00<?, ? examples/s]

Map:   0%|          | 0/486 [00:00<?, ? examples/s]

Dataset ready. Training samples: 3888


In [ ]:
# --- PRE-TOKENIZATION (Crucial for standard Trainer) ---
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=1024, # 4B model can handle 1024 easily
        padding=False
    )

# Ensure you run this on your formatted datasets
tokenized_train = train_dataset.map(tokenize_function, batched=True, remove_columns=["text"])
tokenized_test = test_dataset.map(tokenize_function, batched=True, remove_columns=["text"])

Map:   0%|          | 0/3888 [00:00<?, ? examples/s]

Map:   0%|          | 0/486 [00:00<?, ? examples/s]

**LoRA Configuration**

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules="all-linear",
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 50,499,584 || all params: 7,991,600,416 || trainable%: 0.6319


**The Training Process**

Note on max_seq_length: Since lr-sum contains long articles, I have set it to 1024. If your A100 has 80GB VRAM, you can increase this to 2048 for better results on very long meeting transcripts.


In [ ]:
# Enable gradient checkpointing (saves VRAM, though less critical for 4B)
model.gradient_checkpointing_enable()

# Data Collator handles padding automatically
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir="gemma-4-4b-khmer-sum",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    warmup_steps=50,
    max_steps=500,
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    optim="paged_adamw_8bit",
    save_strategy="steps",
    save_steps=100,
    eval_strategy="steps",
    eval_steps=100,
)

trainer = Trainer(
    model=model,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    args=training_args,
    data_collator=data_collator,
)

# Clear GPU memory and start
torch.cuda.empty_cache()
trainer.train()

Step,Training Loss,Validation Loss
100,1.396029,1.411558
200,1.366349,1.363997
300,1.287051,1.344525
400,1.275446,1.327864
500,1.217436,1.321466


TrainOutput(global_step=500, training_loss=1.3876745052337647, metrics={'train_runtime': 4198.345, 'train_samples_per_second': 1.906, 'train_steps_per_second': 0.119, 'total_flos': 2.21279329714176e+17, 'train_loss': 1.3876745052337647, 'epoch': 2.05761316872428})

**Testing the Long-form Summarizer**

In [ ]:
ARTICLE = """
Hugging Face៖ បដិវត្តន៍នៃដំណើរការភាសាធម្មជាតិ

សេចក្តីផ្តើម
នៅក្នុងវិស័យដំណើរការភាសាធម្មជាតិ (NLP) ដែលកំពុងវិវត្តយ៉ាងឆាប់រហ័ស Hugging Face បានលេចចេញជាកម្លាំងចលករដ៏លេចធ្លោ និងប្រកបដោយភាពច្នៃប្រឌិត។ អត្ថបទនេះនឹងស្វែងយល់ពីដំណើររឿង និងសារៈសំខាន់របស់ Hugging Face ដែលជាក្រុមហ៊ុនមួយបានរួមចំណែកយ៉ាងកត់សម្គាល់ដល់ NLP និង AI ទាំងមូល។ ចាប់តាំងពីការចាប់កំណើតរហូតដល់តួនាទីរបស់ខ្លួនក្នុងការធ្វើឱ្យ AI ក្លាយជាប្រជាធិបតេយ្យ Hugging Face បានបន្សល់ទុកនូវស្លាកស្នាមដែលមិនអាចលុបបាននៅលើឧស្សាហកម្មនេះ។

កំណើតរបស់ Hugging Face
Hugging Face ត្រូវបានបង្កើតឡើងក្នុងឆ្នាំ ២០១៦ ដោយលោក Clément Delangue លោក Julien Chaumond និងលោក Thomas Wolf ។ ឈ្មោះ "Hugging Face" ត្រូវបានជ្រើសរើសដើម្បីឆ្លុះបញ្ចាំងពីបេសកកម្មរបស់ក្រុមហ៊ុនក្នុងការធ្វើឱ្យម៉ូដែល AI កាន់តែងាយស្រួលប្រើប្រាស់ និងរួសរាយរាក់ទាក់ចំពោះមនុស្ស ប្រៀបបាននឹងការឱបក្រសោបដ៏កក់ក្តៅ។ ដំបូងឡើយ ពួកគេបានចាប់ផ្តើមជាក្រុមហ៊ុន Chatbot ប៉ុន្តែក្រោយមកបានផ្លាស់ប្តូរការផ្តោតអារម្មណ៍ទៅលើ NLP ដោយជំរុញដោយជំនឿរបស់ពួកគេលើសក្តានុពលផ្លាស់ប្តូរនៃបច្ចេកវិទ្យានេះ។

ការច្នៃប្រឌិតដែលផ្លាស់ប្តូរវិស័យនេះ
Hugging Face ត្រូវបានគេស្គាល់ច្រើនជាងគេចំពោះការរួមចំណែកកូដបើកចំហរ (Open-Source) របស់ខ្លួន ជាពិសេសបណ្ណាល័យ "Transformers" ។ បណ្ណាល័យនេះបានក្លាយជាស្តង់ដារជាក់ស្តែងសម្រាប់ NLP និងអាចឱ្យអ្នកស្រាវជ្រាវ អ្នកអភិវឌ្ឍន៍ និងអង្គការនានា ងាយស្រួលចូលប្រើប្រាស់ និងប្រើប្រាស់ម៉ូដែលភាសាដែលបានបង្ហាត់រួចជាស្រេចទំនើបៗ ដូចជា BERT, GPT-3 និងម៉ូដែលជាច្រើនទៀត។ ម៉ូដែលទាំងនេះមានកម្មវិធីរាប់មិនអស់ ចាប់ពី Chatbot និងជំនួយការនិម្មិត រហូតដល់ការបកប្រែភាសា និងការវិភាគមនោសញ្ចេតនា។

ការរួមចំណែកសំខាន់ៗ៖
១. បណ្ណាល័យ Transformers៖ បណ្ណាល័យ Transformers ផ្តល់នូវចំណុចប្រទាក់បង្រួបបង្រួមមួយសម្រាប់ម៉ូដែលដែលបានបង្ហាត់រួចជាង ៥០ ដែលជួយសម្រួលដល់ការអភិវឌ្ឍកម្មវិធី NLP ។ វាអនុញ្ញាតឱ្យអ្នកប្រើប្រាស់ធ្វើការបង្ហាត់បន្ថែម (Fine-Tune) ម៉ូដែលទាំងនេះសម្រាប់កិច្ចការជាក់លាក់ ដែលធ្វើឱ្យវាអាចចូលប្រើប្រាស់បានសម្រាប់ទស្សនិកជនកាន់តែទូលំទូលាយ។
២. Model Hub៖ Model Hub របស់ Hugging Face គឺជាឃ្លាំងកំណប់នៃម៉ូដែលដែលបានបង្ហាត់រួចជាស្រេច ដែលធ្វើឱ្យវាងាយស្រួលសម្រាប់អ្នកណាម្នាក់ក្នុងការចូលប្រើប្រាស់ ពិសោធន៍ និងធ្វើការបង្ហាត់បន្ថែមលើម៉ូដែល។ អ្នកស្រាវជ្រាវ និងអ្នកអភិវឌ្ឍន៍នៅជុំវិញពិភពលោកអាចសហការគ្នា និងចែករំលែកម៉ូដែលរបស់ពួកគេតាមរយៈវេទិកានេះ។
៣. សហគមន៍ Hugging Face Transformers៖ Hugging Face បានជំរុញឱ្យមានសហគមន៍អនឡាញដ៏រស់រវើកមួយ ដែលអ្នកអភិវឌ្ឍន៍ អ្នកស្រាវជ្រាវ និងអ្នកចូលចិត្ត AI អាចចែករំលែកចំណេះដឹង កូដ និងការយល់ដឹងរបស់ពួកគេ។ ស្មារតីសហការនេះបានពន្លឿនការរីកចម្រើនរបស់ NLP ។

ការធ្វើឱ្យ AI ក្លាយជាប្រជាធិបតេយ្យ
ផលប៉ះពាល់ដ៏សំខាន់បំផុតរបស់ Hugging Face គឺការធ្វើឱ្យ AI និង NLP ក្លាយជាប្រជាធិបតេយ្យ។ ការប្តេជ្ញាចិត្តរបស់ពួកគេចំពោះការអភិវឌ្ឍកូដបើកចំហរ បានធ្វើឱ្យម៉ូដែល AI ដ៏មានអានុភាពអាចចូលប្រើប្រាស់បានសម្រាប់បុគ្គល ក្រុមហ៊ុនចាប់ផ្តើមថ្មី (Startups) និងអង្គការដែលបានបង្កើតឡើងរួចហើយ។ វិធីសាស្រ្តនេះផ្ទុយស្រឡះពីទីផ្សារម៉ូដែល AI កម្មសិទ្ធិបែបប្រពៃណី ដែលជារឿយៗដាក់កម្រិតការចូលប្រើប្រាស់ចំពោះតែអ្នកដែលមានធនធានច្រើនប៉ុណ្ណោះ។

តាមរយៈការផ្តល់នូវម៉ូដែល និងឧបករណ៍កូដបើកចំហរ Hugging Face បានផ្តល់អំណាចដល់អ្នកប្រើប្រាស់ចម្រុះជាច្រើនឱ្យច្នៃប្រឌិត និងបង្កើតកម្មវិធី NLP ផ្ទាល់ខ្លួនរបស់ពួកគេ។ ការផ្លាស់ប្តូរនេះបានជំរុញឱ្យមានបរិយាបន្ន ដោយអនុញ្ញាតឱ្យមានសំឡេងកាន់តែទូលំទូលាយក្នុងការរួមចំណែកដល់ការស្រាវជ្រាវ និងអភិវឌ្ឍន៍ AI ។

ការទទួលយកដោយឧស្សាហកម្ម
ភាពជោគជ័យ និងផលប៉ះពាល់របស់ Hugging Face គឺបង្ហាញឱ្យឃើញតាមរយៈការទទួលយកយ៉ាងទូលំទូលាយរបស់ខ្លួន។ ក្រុមហ៊ុន និងស្ថាប័នជាច្រើន ចាប់ពីក្រុមហ៊ុនចាប់ផ្តើមថ្មីរហូតដល់ក្រុមហ៊ុនបច្ចេកវិទ្យាយក្ស បានប្រើប្រាស់បច្ចេកវិទ្យារបស់ Hugging Face សម្រាប់កម្មវិធី AI របស់ពួកគេ។ នេះរួមបញ្ចូលទាំងឧស្សាហកម្មចម្រុះដូចជា ការថែទាំសុខភាព ហិរញ្ញវត្ថុ និងការកម្សាន្ត ដែលបង្ហាញពីភាពបត់បែនរបស់ NLP និងការរួមចំណែករបស់ Hugging Face ។

ទិសដៅអនាគត
ដំណើររបស់ Hugging Face គឺនៅឆ្ងាយពីការបញ្ចប់នៅឡើយ។ យោងតាមចំណេះដឹងចុងក្រោយរបស់ខ្ញុំនៅខែកញ្ញា ឆ្នាំ២០២១ ក្រុមហ៊ុនកំពុងស្វែងរកយ៉ាងសកម្មនូវការស្រាវជ្រាវទៅលើ AI ប្រកបដោយក្រមសីលធម៌ ការកាត់បន្ថយភាពលម្អៀងនៅក្នុងម៉ូដែល និងច្រើនទៀត។ ដោយផ្អែកលើកំណត់ត្រានៃការច្នៃប្រឌិត និងការប្តេជ្ញាចិត្តរបស់ពួកគេចំពោះសហគមន៍ AI វាទំនងជាថាពួកគេនឹងបន្តដឹកនាំក្នុងការអភិវឌ្ឍ AI ប្រកបដោយក្រមសីលធម៌ និងលើកកម្ពស់ការប្រើប្រាស់បច្ចេកវិទ្យា NLP ប្រកបដោយការទទួលខុសត្រូវ។

សេចក្តីសន្និដ្ឋាន
ដំណើររឿងរបស់ Hugging Face គឺជារឿងមួយនៃការផ្លាស់ប្តូរ ការសហការ និងការផ្តល់សិទ្ធិអំណាច។ ការរួមចំណែកកូដបើកចំហរបស់ពួកគេបានផ្លាស់ប្តូរទិដ្ឋភាពរបស់ NLP និងធ្វើឱ្យ AI ក្លាយជាប្រជាធិបតេយ្យ។ នៅពេលដែលពួកគេបន្តរុញច្រានព្រំដែននៃការស្រាវជ្រាវ AI យើងអាចរំពឹងថា Hugging Face នឹងបន្តស្ថិតនៅជួរមុខនៃការច្នៃប្រឌិត ដោយរួមចំណែកដល់អនាគត AI ដែលកាន់តែមានបរិយាបន្ន និងប្រកបដោយក្រមសីលធម៌។ ដំណើររបស់ពួកគេរំឭកយើងថា អំណាចនៃការសហការបែបកូដបើកចំហរ អាចនាំទៅរកភាពជឿនលឿនដ៏អស្ចារ្យក្នុងបច្ចេកវិទ្យា និងនាំ AI ឱ្យទៅដល់ដៃមនុស្សជាច្រើន។
"""

In [ ]:
def summarize_long_khmer(text):
    # The prompt remains the same (Correct for Gemma-4-it)
    prompt = f"<start_of_turn>user\nYou are a high-accuracy Khmer summarization assistant. Summarize the following text in Khmer with concise, factual language.\n\n{text}<end_of_turn>\n<start_of_turn>model\n"

    # ADDED: truncation=True and max_length to prevent OOM crashes with long articles
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to("cuda")

    # Generate
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.3,
        do_sample=True,
        eos_token_id=tokenizer.eos_token_id # Ensures the model stops when finished
    )

    # SLICE the tokens instead of splitting the string
    # This removes exactly the number of tokens that were in the input prompt
    input_length = inputs.input_ids.shape[1]
    generated_tokens = outputs[0][input_length:]

    # Decode only the newly generated part
    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

# Test with your article
print(summarize_long_khmer(ARTICLE))

ាចិត្តរបស់ក្រុមហ៊ុនក្នុងការធ្វើឱ្យម៉ូដែល AI ក្លាយជាកូដបើកចំហរ មិនត្រឹមតែផ្តល់ឱ្យអ្នកស្រាវជ្រាវ និងអ្នកអភិវឌ្ឍន៍នូវឧបករណ៍ដ៏មានឥទ្ធិពលមួយប៉ុណ្ណោះទេ ប៉ុន្តែថែមទាំងផ្តល់ឱ្យក្រុមហ៊ុនធំៗ និងអង្គការក្រៅរដ្ឋាភិបាលនូវឧបករណ៍ទាំងនេះផងដែរ។ នេះបានធ្វើឱ្យការស្រាវជ្រាវ និងការអភិវឌ្ឍន៍ម៉ូដែល AI កាន់តែមានល្បឿនលឿន និងមានការចូលរួមពីសាធារណជនកាន់តែច្រើន។ ក្រុមហ៊ុនធំៗដូចជា Google និង Microsoft បានចាប់ផ្តើមប្រើប្រាស់បណ្ណាល័យ Transformers របស់ Hugging Face សម្រាប់គម្រោងរបស់ពួកគេ។ នេះបានធ្វើឱ្យបណ្ណាល័យនេះក្លាយជាស្តង់ដារមួយសម្រាប់អ្នកអភិវឌ្ឍន៍។ ក្រុមហ៊ុនធំៗទាំងនេះក៏បានប្រើប្រាស់ Model Hub របស់ Hugging Face ដើម្បីចែករំលែកម៉ូដែលរបស់ពួកគេផងដែរ។ នេះបានធ្វើឱ្យការស្រាវជ្រាវ និងការអភិវ


In [ ]:
model.save_pretrained("/content/drive/MyDrive/gemma-4-4b-khmer-sum")
tokenizer.save_pretrained("/content/drive/MyDrive/gemma-4-4b-khmer-sum")

('/content/drive/MyDrive/gemma-4-4b-khmer-sum/tokenizer_config.json',
 '/content/drive/MyDrive/gemma-4-4b-khmer-sum/chat_template.jinja',
 '/content/drive/MyDrive/gemma-4-4b-khmer-sum/tokenizer.json')

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base_model_id = smaller_model_id
adapter_path = "/content/drive/MyDrive/gemma-4-4b-khmer-sum"
hf_repo_name = "lonewolf168/gemma-4-4b-khmer-sum"

print("Step 1: Loading Base Model on CPU... (This takes time)")
# We load in bfloat16 on CPU and use low_cpu_mem_usage to prevent RAM crashes
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    device_map="cpu",
    trust_remote_code=True
)

print("Step 2: Loading Adapters...")
model = PeftModel.from_pretrained(base_model, adapter_path)

print("Step 3: Merging Weights... (This is the magic part)")
# This permanently combines the LoRA weights with the base weights
fused_model = model.merge_and_unload()

print("Step 4: Loading Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(base_model_id)

Step 1: Loading Base Model on CPU... (This takes time)


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

Step 2: Loading Adapters...
Step 3: Merging Weights... (This is the magic part)
Step 4: Loading Tokenizer...


In [ ]:
print("Step 5a: Saving Fused Model to Google Drive...")
save_path = "/content/drive/MyDrive/gemma-4-4b-khmer-lr-sum-fused"
fused_model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"✅ Fused model saved to {save_path}")

print(f"Step 5b: Pushing to Hugging Face Hub: {hf_repo_name}...")
fused_model.push_to_hub(hf_repo_name)
tokenizer.push_to_hub(hf_repo_name)

print("✅ DONE! Your production model is now live on Hugging Face.")

Step 5a: Saving Fused Model to Google Drive...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Fused model saved to /content/drive/MyDrive/gemma-4-4b-khmer-lr-sum-fused
Step 5b: Pushing to Hugging Face Hub: lonewolf168/gemma-4-4b-khmer-sum...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

...

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mps1rbaq4_/tokenizer.json:  24%|##4       | 7.88MB / 32.2MB            

✅ DONE! Your production model is now live on Hugging Face.


In [ ]:
def get_prompt(text):
    prompt = (
    "<start_of_turn>user\n"
    "You are an expert Khmer Synthesis Assistant. Your task is to analyze the provided text and "
    "provide the most appropriate professional summary in Khmer based on the content type.\n\n"

    "### INSTRUCTIONS:\n"
    "1. If the text is a MEETING TRANSCRIPT:\n"
    "   - Format the output as 'Professional Meeting Minutes'.\n"
    "   - Use these sections: 🎯 គោលបំណងនៃកិច្ចប្រជុំ (Objective), 💬 ចំណុចពិភាក្សាគន្លឹះ (Key Points), "
    "✅ សេចក្តីសម្រេច (Decisions), and 📝 កិច្ចការដែលត្រូវបន្ត (Action Items).\n\n"

    "2. If the text is a GENERAL DOCUMENT/ARTICLE:\n"
    "   - Format the output as a 'High-Accuracy Summary'.\n"
    "   - Provide a concise overview followed by a bulleted list of the most important factual points.\n\n"

    "### GUIDELINES:\n"
    "- Use formal, professional Khmer language.\n"
    "- Be factual, concise, and objective.\n"
    "- Do not add information that is not present in the text.\n\n"

    f"TEXT TO PROCESS:\n{text}\n"
    "<end_of_turn>\n"
    "<start_of_turn>model\n"
)
    return prompt

In [ ]:
fused_model.to("cuda")
fused_model.eval() # Set to evaluation mode for consistent results

def summarize_long_khmer(text):
    # Prompt with dedented formatting to avoid leading whitespace/tabs
    # We use a clean string to ensure the model doesn't see indentation spaces
    prompt = get_prompt(text)

    # Tokenize with truncation to prevent OOM
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to("cuda")

    # Generate with increased token limit for structured output
    with torch.no_grad(): # Disable gradient calculation for faster inference
        outputs = fused_model.generate(
            **inputs,
            max_new_tokens=512, # INCREASED: To fit all 4 sections of the minutes
            temperature=0.2,     # LOWERED: More factual and stable for minutes
            do_sample=True,
            eos_token_id=tokenizer.eos_token_id
        )

    # Slice tokens to remove the input prompt
    input_length = inputs.input_ids.shape[1]
    generated_tokens = outputs[0][input_length:]

    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

# Test with article
print(summarize_long_khmer(ARTICLE))

។ ក្រុមហ៊ុននេះបានចាប់ផ្តើមដោយការផ្តល់នូវបណ្តុំម៉ូដែលដែលបានហ្វឹកហាត់រួចស្រេច (pre-trained models) សម្រាប់កិច្ចការ NLP ផ្សេងៗ។ ម៉ូដែលទាំងនេះត្រូវបានបង្កើតឡើងដោយក្រុមអ្នកស្រាវជ្រាវ និងអ្នកអភិវឌ្ឍន៍របស់ Hugging Face ផ្ទាល់។ ម៉ូដែលទាំងនេះត្រូវបានចែករំលែកដោយឥតគិតថ្លៃ តាមរយៈបណ្តាញទូទៅមួយដែលគេហៅថា Hugging Face Hub។ នេះគឺជាបណ្តាញទូទៅមួយដែលអនុញ្ញាតឱ្យអ្នកប្រើប្រាស់ចែករំលែកម៉ូដែល និងទិន្នន័យរបស់ពួកគេ។ ម៉ូដែលទាំងនេះមានប្រភពមកពីបណ្តុំម៉ូដែលដ៏ធំមួយដែលមានឈ្មោះថា Transformers ដែលត្រូវបានបង្កើតឡើងដោយក្រុមអ្នកស្រាវជ្រាវរបស់ Hugging Face ផ្ទាល់។ ម៉ូដែល Transformers នេះបានក្លាយទៅជាស្តង់ដារថ្មីមួយសម្រាប់កិច្ចការ NLP នានា។ ម៉ូដែល Transformers នេះបានផ្តល់នូវការកែប្រែយ៉ាងច្រើនដល់ម៉ូដែលបច្ចុប្បន្ន ដូចជាការកែប្រែម៉ូដែល BERT របស់ Google ឱ្យមានប្រសិទ្ធភាពជាងមុន និងមានលទ្ធភាពប្រើប្រាស់បានទូលំទូលាយជាងមុន។ ម៉ូដែល Transformers នេះក៏បានផ្តល់នូវការកែប្រែយ៉ាងច្រើនដល់ម៉ូដែល GPT-2 របស់ OpenAI ផងដែរ។ ម៉ូដែល Transformers នេះក៏បានផ្តល់នូវការកែប្រែយ៉ាងច្រើនដល់ម៉ូដែល RoBERTa របស់ Facebook ផងដែរ។ ម៉ូដែល Transformers នេះបានក្លាយ

In [ ]:
ARTICLE1 = """ ក្រុមហ៊ុន Meta ដែលជាម្ចាស់ Facebook និង Instagram កំពុងធ្វើឱ្យពិភពបច្ចេកវិទ្យាមានការភ្ញាក់ផ្អើលជាថ្មី ដោយគ្រោងនឹងបញ្ឈប់បុគ្គលិកទ្រង់ទ្រាយធំក្នុងរលកទីមួយនៅថ្ងៃទី ២០ ខែឧសភាខាងមុខនេះ។ បើយោងតាមរបាយការណ៍ពីទីភ្នាក់ងារសារព័ត៌មាន Reuters ចេញផ្សាយនៅថ្ងៃទី១៨ ខែមេសា ឆ្នាំ២០២៦​ បានឱ្យដឹងថា បុគ្គលិកប្រហែល ៨,០០០ នាក់ (ស្មើនឹង ១០% នៃបុគ្គលិកសរុប) នឹងត្រូវបាត់បង់ការងារក្នុងដំណាក់កាលដំបូងនេះ ហើយការកាត់បន្ថយនឹងបន្តធ្វើឡើងបន្ថែមទៀតរហូតដល់ចុងឆ្នាំ ២០២៦។

មូលហេតុចម្បងនៃការកាត់បន្ថយនេះ គឺដោយសារលោក Mark Zuckerberg កំពុងសម្រុកបោះទុនរាប់ពាន់លានដុល្លារលើបច្ចេកវិទ្យាបញ្ញាសិប្បនិម្មិត (AI) ដោយលោកមានគោលដៅចង់ «ផ្លាស់ប្តូរទម្រង់ការងារផ្ទៃក្នុងរបស់ក្រុមហ៊ុនយ៉ាងខ្លាំងក្លា»។ ថ្នាក់ដឹកនាំ Meta យល់ឃើញថា ការប្រើប្រាស់ AI មកជួយការងារ នឹងធ្វើឱ្យក្រុមហ៊ុនដើរលឿនជាងមុន និងមានប្រសិទ្ធភាពជាងមុន ដោយមិនចាំបាច់មានបុគ្គលិកច្រើនជាន់ថ្នាក់ស្មុគស្មាញដូចមុនឡើយ។ ជាក់ស្តែង វិស្វករជាច្រើនត្រូវបានផ្ទេរទៅកាន់ផ្នែក «Applied AI» ដើម្បីបង្កើតប្រព័ន្ធឆ្លាតវៃដែលអាចសរសេរកូដ និងដោះស្រាយការងារស្មុគស្មាញដោយស្វ័យប្រវត្តិជំនួសមនុស្ស។

និន្នាការ «យក AI ជំនួសមនុស្ស» នេះ កំពុងក្លាយជា «គំរូទូទៅក្នុងចំណោមក្រុមហ៊ុនធំៗនៅសហរដ្ឋអាមេរិក»។ មិនមែនមានតែ Meta ទេ សូម្បីតែក្រុមហ៊ុនយក្ស Amazon និងក្រុមហ៊ុនហិរញ្ញវត្ថុ Block ក៏បានបញ្ឈប់បុគ្គលិករាប់ម៉ឺននាក់រួចទៅហើយ ដោយយោងលើហេតុផលដូចគ្នាគឺ៖ យក AI មកធ្វើការដើម្បីចំណេញពេលវេលា និងថវិកា។

គួរជម្រាបថា បើទោះបីជាកាលពីឆ្នាំមុន Meta រកចំណេញបានរហូតដល់ ៦០ ពាន់លានដុល្លារ និងមានស្ថានភាពហិរញ្ញវត្ថុរឹងមាំក៏ដោយ ប៉ុន្តែយុទ្ធសាស្ត្រថ្មីនេះបានបង្ហាញយ៉ាងច្បាស់ថា នៅក្នុងយុគសម័យបច្ចេកវិទ្យាថ្មី កម្លាំងមនុស្សលែងជាអាទិភាពទីមួយទៀតហើយ ពោលគឺ AI នឹងក្លាយជាអ្នកក្តាប់វាសនានៃការងារនាពេលអនាគតវិញម្តង៕ """

ARTICLE2 = """ ចរាចរណ៍​នៅ​ច្រក​អកមូហ្ស​មាន​សកម្មភាព​មិនទាន់​​បាន​មួយ​ថ្ងៃ​ផង អ៊ីរ៉ង់​បាន​ប្រកាស​បិទ​ច្រក​សមុទ្ទ​អកមូហ្ស​វិញ នៅ​ព្រឹក​ថ្ងៃសៅរ៍​ទី​១៨​មេសា ដោយ​ពន្យល់​ថា អ៊ីរ៉ង់​ខំ​​មាន​ចេតនា​ល្អ​ព្រម​ឱ្យ​នាវា​ស៊ីវិល​ធ្វើ​ចរាចរណ៍​ឆ្លង​កាត់​ច្រក​អកមូហ្ស​ឡើង​វិញ តែអាមេរិក​បែរ​ជា​រំលោភ​ពាក្យ​សន្យា។ អាមេរិក​​បន្ត​បិទ​ខ្ទប់​កំពង់​ផែ​អ៊ីរ៉ង់​ដដែល។

ស្ថានភាព​ច្រក​សមុទ្ទ​អកមូហ្ស​ត្រូវ​វិល​ទៅ​រក​សភាព​ដើម ពោល​គឺ អ៊ីរ៉ង់សម្រេច​បិទ​ច្រក​យុទ្ធសាស្ត្រនេះវិញ និង​ចាប់​ផ្តើម​គ្រប់គ្រង​​ រាល់​សកម្មភាព ចេញ​ចូល​ របស់គ្រប់​​នាវា​​ យ៉ាង​តឹង​តែងបំផុត។ ដោយហេតុ​តែ​អាមេរិក​នៅ​តែ​បន្ត​បិទ​ខ្ទប់​តំបន់​ច្រក​សមុទ្ទ​អកមូហ្ស ទើបអ៊ីរ៉ង់ប្តូរ​ចិត្ត​ ត្រលប់​ទៅបិទ​ច្រក​សមុទ្ទ​អកមូហ្ស​វិញ។ នេះ​បើ​តាម​សេចក្តី​ប្រកាស​របស់ប្រមុខ​ការទូត​អ៊ីរ៉ង់ ដោយ​​​ប្រកាសថា​គ្មាន​ការ​ចរចា​អាមេរិក​អ៊ីរ៉ង់ ជុំ​ទីពីរ នៅ​ប៉ាគីស្ថាន​ទេ។

​​សូម​បញ្ជាក់​ថា នៅ​រសៀល​ថ្ងៃសុក្រ​ទី ១៧​មេសា​ម្សិលមិញ​ បន្ទាប់ពីប្រធានាធិបតី​អាមេរិក​ប្រកាស​បទឈប់បាញ់រយៈពេល​១០ថ្ងៃ រវាង​អ៊ីស្រាអែល និង​លីបង់ភ្លាម រដ្ឋមន្រ្តីការបរទេស​អ៊ីរ៉ង់​លោក​អាបាស អារ៉ាឈី បាន​ប្រកាសបើក​ច្រក​អកមូហ្សភ្លែត។ តែ​ក៏​មាន​​ព័ត៌មាន​ចម្រូង​ចម្រាសគ្នា​ច្រើន​ មិន​គួរ​ឱ្យ​ទុក​ចិត្ត​​ដែរ។ ជា​សរុប​ មាន​នាវា​ដឹក​ប្រេង និង​ឧស្ម័ន ​ប្រមាណ​តែ​៨​គ្រឿងប៉ុណ្ណោះ បាន​ឆ្លង​កាត់​ច្រក​អកមូហ្ស​​រួច ក្នុង​រយៈ​ពេលនៃ​ការ​បើក​ច្រក​បាន​​ជិត​២០ម៉ោង៕​ """

ARTICLE3 = """យោធា​អ៊ីស្រាអែល បាន​ប្រកាស​នៅ​ថ្ងៃ​សៅរ៍​ទី​១៨​មេសា​២០២៦​នេះ គូស​បន្ទាត់​ “​លឿង” បែង​ចែក​តំបន់​ភាគ​ខាង​ត្បូង​​​លី​បង់​ជា​ច្រើន​ចំរៀក តាម​គំរូ​ដែល​អ៊ីស្រាអែល​ធ្លាប់​អនុវត្ត​នៅ​តំបន់​ហ្កាហ្សា​របស់​ប៉ាឡេស្ទីន។ ពោល​គឺ​អ៊ីស្រាអែល​​បាន​ពុះ​ចែក​ភូមិ​ឋាន​ប្រជាជន ដើម្បី​បង្កើត​ជា​តំបន់​យោធា​ ដែល​ស្ថិត​នៅ​ក្រោម​ការ​ត្រួត​ពិនិត្យ​យ៉ាង​តឹងរ៉ឹង​ និង​ដើម្បី​រារាំង​កុំ​ឱ្យ​ចលនា​ហេសបូឡា​មាន​សមត្ថភាព​បង្ក​បង្កើត​ជា​ក្រុម​ ផ្តុំ​ជា​​កម្លាំង​បាន។ នេះ​បើ​តាម​ប្រភព​យោធា​អ៊ីស្រាអែល​​ដោយ​បន្ថែម​ទៀត​ថា តាំងពី​មាន​គំនូស​​ខ្សែបន្ទាត់​លឿង ​បាន​២៤​ម៉ោងមក កងកម្លាំងអ៊ីស្រាអែល​កំណត់​ឃើញ​មាន​ភេរវជន​រំលោភបទឈប់​បាញ់ ហើយ​រំកិល​ខ្លួន​មក​ជិត​​ខ្សែ​បន្ទាត់​ហាម​ឃាត់​ច្រើន តែ​យោធា​អ៊ីស្រាអែល​ក៏​បាន​កម្ចាត់​​ការ​គំរាម​កំហែង​ដោយ​ជាក់​ស្តែង​នោះ​យ៉ាង​ទាន់​ហន់​វិញ​ដែរ។

“ខ្សែ​បន្ទាត់​លឿង” ដែល​ជារបៀប​គូស​​​ចែកតំបន់ មិន​ត្រឹម​តែជា​​បន្ទាត់​ហាមឃាត់​មិន​ឱ្យ​ជន​ស៊ីវិល​ភៀស​ខ្លួន​លីបង់​អាច​ត្រលប់​មក​កាន់​ភូមិកំណើតឬ​លំនៅ​ឋាន​​ខ្លួនវិញទេ​​ តែ​ជា​ការ​បង្កើត​តំបន់​យោធា ដែលអនុញ្ញាត​ឱ្យ​​ទ័ព​អ៊ីស្រាអែល​អាច​រក្សា​សេរីភាពផ្នែក​​ធ្វើ​ប្រតិបត្តិការ​យោធា​ផ្សេងៗ នៅក្នុង​តំបន់​ និង​ដោយ​គ្មាន​​ល្មើស​នឹង​​​បទ​ឈប់​បាញ់ទៀត។ តាម​រយៈ​យុទ្ធសាស្ត្រ​បង្កើត​ខ្សែ​បន្ទាត់​លឿងនេះ ទោះ​បីមាន​​បទ​ឈប់​បាញ់​អ៊ីស្រាអែល-​លីបង់ ​ដ៏​ផុយ​ស្រួយយ៉ា​ង​ណា​ក៏​ដោយ ក៏​អ៊ីស្រាអែល​អាច​ការពារ​ព្រំដែន​​ ផ្នែក​ខាង​ជើង​ប្រទេស​ពី​ការ​ជ្រៀត​ចូល​របស់​ភេរវជន​ផង និង​​ក្រុម​អង្គការ​ប្រដាប់​អាវុធ​ផង។

និយាយ​ពី​​បន្ទាត់​លឿង​ហាមឃាត់​ជា​លើក​ទីមួយ នៅក្នុង​សេចក្តី​ថ្លែងការណ៍ ​យោធា​អ៊ីស្រាអែលបាន​អះអាងទៀត​​ថា ទ័ព​អាកាសនិង​ទ័ព​ថ្មើរជើងអ៊ីស្រាអែល ព្រមទាំង​កង​កាំភ្លើង​ធំ បានធ្វើ​ប្រតិបត្តិការ​​បាញ់​លើ​ពួក​ភេរវជន​សង្ស័យ​ នៅ​ជា​ច្រើន​កន្លែង​ក្នុងតំបន់​ភាគ​ខាង​ត្បូង​​លីបង់ ក្នុង​ថ្ងៃ​ទី​ពីរ​នៃ​បទ​ឈប់​បាញ់​ដែលទើប​​ចូល​ជា​ធរមាននៅថ្ងៃ​សុក្រ​ទី​១៧​មេសា​សម្រាប់​រយៈ​ពេល​១០​ថ្ងៃ។

ប៉ុន្តែ​បទ​ឈប់​បាញ់ និង​គម្រោង​ចរចា​សន្តិភាព​ដោយ​ផ្ទាល់​រវាង​​អ៊ីស្រាអែល​លីបង់​ ​ត្រូវ​បាន​ចលនា​ហេសបូឡា​លីបង់​ច្រាន​ចោល​ទាំង​ស្រុង​នៅ​ថ្ងៃ​សៅរ៍​ទី​១៨មេសា​នេះ។ គឺ​ក្នុង​បរិបទ​នេះស្រាប់​តែ​​មាន​ការ​វាយប្រហារ​ពី​សំណាក់​ក្រុម​ប្រដាប់អាវុធ​មិន​ស្គាល់​អត្តសញ្ញាណដែល​បើតាម​ការ​សង្ស័យ​ជា​ក្រុម​ហេសបូឡា​ បណ្តាល​ឱ្យ​កងទ័ព​មួកខៀវអង្គការ​សហប្រជាជាតិ ជាតិ​បារាំង​ ​​ម្នាក់​ស្លាប់​ និង​បី​នាក់​ទៀត​រង​របួស។ ប្រធានាធិបតីបារាំង​បាន​ទាមទារ​ឱ្យ​​លីបង់ធានា​រក្សា​សុវត្ថិភាព​​ជូន​កង​កម្លាំង​អង្គការ​សហប្រជាជាតិ។ ប្រធាន​សភា​លីបង់​ដែល​ស្និទ្ធ​នឹង​ចលនា​ហេសបូឡា​ ក៏​បាន​ចេញ​មុខ​ថ្កោល​ទោស​ការ​វាយប្រហារ​លើ​កងទ័ព​មួក​ខៀវ។ ចំណែក​ប្រធានាធិបតី​លីបង់ ​សន្យា​​មិន​នៅ​ស្ងៀម​បណ្តោយ​ឱ្យ​ជន​ដៃ​ដល់​រួច​ខ្លួន​ឡើយ៕ """

ARTICLE4 = """ តើមាសកម្រិត 10K, 14K, 18K, និង 24K ខុសគ្នាបែបណាដែរ?

នេះជាចំណេះដឹងទូទៅមួយដ៏មានសារៈសំខាន់ ហើយ Fresh News សូមចែករំលែកចំណេះដឹងនេះ ដូចខាងក្រោម៖

* មាសកម្រិត 24K៖

24K គឺជាមាសសុទ្ធតែម្តង ឬបង្ហាញថា មាសនោះសុទ្ធចាប់ពី 99.95% ឡើងទៅ។ គួរកត់សម្គាល់ថា ភាគច្រើនយើងមិនដែលឃើញគេយកមាសកម្រិតនេះ ទៅធ្វើជាគ្រឿងអលង្ការឡើយ ដោយសារមាសសុទ្ធមានលក្ខណៈទន់ និងងាយនឹងរងការឆ្កូតណាស់។ ដូច្នេះហើយបានជារឿយៗ គេតែងតែបញ្ចូលលោហៈធាតុផ្សេងទៀតទៅក្នុងមាស ដើម្បីឲ្យវារឹង និងជាប់បានយូរ ដើម្បីធ្វើជាគ្រឿងអលង្ការ។

* មាសកម្រិត 18K៖

18K គឺជាមាសដែលត្រូវបានបង្កើតឡើងដោយផ្ទុកមាស 75% និងមានលោហៈធាតុ 25% ។ មាសប្រភេទនេះ ភាគច្រើនគេសម្គាល់ថា ជាទម្រង់មាសសុទ្ធ ដែលសម្រាប់យកទៅធ្វើជាចិញ្ចៀន នាឡិកា និងគ្រឿងអលង្ការជាច្រើនទៀត។ ទោះបីជា 18K មើលទៅស្រស់ស្អាត ក៏ប៉ុន្តែវាថ្លៃ ហើយក៏ងាយរងការឆ្កូតស្ទើរតែដូចមាសកម្រិត 24K ដែរ។

* មាសកម្រិត 14K៖

14K គឺជាមាសដែលត្រូវបានបង្កើតឡើងដោយមាស 58.3% និងលោហៈធាតុ 41.7% ។ មាសប្រភេទនេះ ជាទូទៅត្រូវបានគេឃើញថា មានប្រើប្រាស់យ៉ាងច្រើនទៅលើគ្រឿងអលង្ការ ដែលលក់នៅប្រទេសអ្នកមាន ដូចជា អាមេរិក អង់គ្លេស និងជាពិសេសប្រទេសលោកខាងលិច។ 14K គឺជាប្រភេទមាសមួយ ដែលមិនងាយឆ្កូត និងរឹងមាំ ប៉ុន្តែថ្លៃជាងកម្រិត 10K បន្តិច។

* មាសកម្រិត 10K៖

10K គឺជាប្រភេទមាសដែលត្រូវបានបង្កើតឡើងដោយមាស 41.7% និងលោហៈធាតុ 58.3% ។ មាសប្រភេទនេះ ធូរថ្លៃជាងគេបំផុត រឹងមាំជាងគេបំផុត ប៉ុន្តែគេហៅវាជាទូទៅថា ជាមាសអន់បំផុតឡើយ ហើយសម្រាប់អ្នកមានលំដាប់ខ្ពស់ គេមិនដែលទិញមាសប្រភេទនេះ ទៅធ្វើជាគ្រឿងអលង្ការឡើយ។ ទោះបីជាវាធូរថ្លៃសម្រាប់អ្នកក្រ និងរឹងមាំជាងកម្រិតមាសផ្សេងទៀត ប៉ុន្តែពណ៌វាមិនសូវស្អាត ហើយងាយនឹងប្រតិកម្មជាមួយស្បែកមនុស្សផងដែរ។

សូមបញ្ជាក់ថា K មានន័យថា ការ៉ាត់ (Karat) សម្រាប់ហៅមាស។ រីឯ ការ៉ាត់ (Carat) ដែលសរសេរដោយអក្សរ C នៅពីមុខ គេសម្គាល់ដល់រង្វាស់រង្វាល់ទម្ងន់ សម្រាប់ហៅពេជ្រ៕ """

In [ ]:
import re

def clean_gemma_output(text):
    """
    Removes Gemma control tokens and cleans up resulting whitespace.
    Works for <start_of_turn>, <start_of_turn>model, <start_of_turn>user, and <end_of_turn>.
    """
    if not text:
        return ""

    # Define the patterns to remove
    # The '|' symbol acts as an 'OR' operator in Regex
    # We list the longest patterns first (e.g., <start_of_turn>model)
    # so they are caught before the shorter ones (e.g., <start_of_turn>)
    patterns = [
        r"<start_of_turn>model",
        r"<start_of_turn>user",
        r"<start_of_turn>",
        r"<end_of_turn>"
    ]

    combined_pattern = "|".join(patterns)

    cleaned_text = re.sub(combined_pattern, "", text)

    cleaned_text = cleaned_text.strip()
    cleaned_text = re.sub(r'\n{3,}', '\n\n', cleaned_text)

    return cleaned_text


In [ ]:
# Test with articles
print("ARTICLE1", clean_gemma_output(summarize_long_khmer(ARTICLE1)))

ARTICLE1 ៃក្នុង» របស់ក្រុមហ៊ុន។ ក្រុមហ៊ុន Meta និយាយថា ខ្លួនបានកាត់បន្ថយបុគ្គលិកប្រមាណ ៨,០០០ នាក់ ក្នុងពេលថ្មីៗនេះ ក្នុងគោលបំណងកាត់បន្ថយការចំណាយ និងធ្វើឱ្យក្រុមហ៊ុនមានប្រសិទ្ធភាពជាងមុន។ ក្រុមហ៊ុនបាននិយាយថា ខ្លួននឹងបន្តកាត់បន្ថយបុគ្គលិកបន្ថែមទៀតរហូតដល់ចុងឆ្នាំ ២០២៦។ ក្រុមហ៊ុន Meta និយាយថា ខ្លួនបានកាត់បន្ថយបុគ្គលិកប្រមាណ ៨,០០០ នាក់ ក្នុងពេលថ្មីៗនេះ ក្នុងគោលបំណងកាត់បន្ថយការចំណាយ និងធ្វើឱ្យក្រុមហ៊ុនមានប្រសិទ្ធភាពជាងមុន។ ក្រុមហ៊ុន Meta និយាយថា ខ្លួននឹងបន្តកាត់បន្ថយបុគ្គលិកបន្ថែមទៀតរហូតដល់ចុងឆ្នាំ ២០២៦។ ក្រុមហ៊ុន Meta និយាយថា ខ្លួនបានកាត់បន្ថយបុគ្គលិកប្រមាណ ៨,០០០ នាក់ ក្នុងពេលថ្មីៗនេះ ក្នុងគោលបំណងកាត់បន្ថយការចំណាយ និងធ្វើឱ្យក្រុមហ៊ុនមានប្រសិទ្ធភាពជាងមុន។ ក្រុមហ៊ុន Meta និយាយថា ខ្លួននឹងបន្តកាត់បន្ថយបុគ្គលិកបន្ថែមទៀតរហូតដល់ចុងឆ្នាំ ២០២៦។ ក្រុមហ៊ុន Meta និយាយថា ខ្លួនបានកាត់បន្ថយបុគ្គលិកប្រមាណ ៨,០០០ នាក់ ក្នុងពេលថ្មីៗនេះ ក្នុងគោលបំណងកាត់បន្ថយការចំណាយ និងធ្វើឱ្យក្រុមហ៊ុនមានប្រសិទ្ធភាពជាងមុន។ ក្រុមហ៊ុន Meta និយាយថា ខ្លួននឹងបន្តកាត់បន្ថយបុគ្គលិកបន្ថែមទៀតរហូតដល់ចុងឆ្នាំ ២០២៦។ ក្រុមហ៊ុន Meta និយាយថា 

In [ ]:
print("ARTICLE2", clean_gemma_output(summarize_long_khmer(ARTICLE2)))

ARTICLE2 ​ការ​ឆ្លង​កាត់​របស់​នាវា​ក្នុង​តំបន់​នោះ។ នេះ​បើ​តាម​ការ​ប្រកាស​របស់​ក្រសួង​ការបរទេស​អ៊ីរ៉ង់​កាលពី​ថ្ងៃ​ពុធ។ ក្រសួង​ការបរទេស​អ៊ីរ៉ង់​បាន​បញ្ជាក់​ថា អាមេរិក​បាន​រំលោភ​ពាក្យ​សន្យា​របស់​ខ្លួន​ដែល​បាន​ផ្តល់​សេចក្តី​ធានា​ថា នឹង​អនុញ្ញាត​ឱ្យ​នាវា​ស៊ីវិល​ធ្វើ​ចរាចរណ៍​ឆ្លង​កាត់​ច្រក​សមុទ្រ​អកមូហ្ស​ឡើង​វិញ។ ក្រសួង​ការបរទេស​អ៊ីរ៉ង់​បាន​បញ្ជាក់​ថា៖ «អាមេរិក​បាន​រំលោភ​ពាក្យ​សន្យា​របស់​ខ្លួន​ដែល​បាន​ផ្តល់​សេចក្តី​ធានា​ថា នឹង​អនុញ្ញាត​ឱ្យ​នាវា​ស៊ីវិល​ធ្វើ​ចរាចរណ៍​ឆ្លង​កាត់​ច្រក​សមុទ្រ​អកមូហ្ស​ឡើង​វិញ។ អាមេរិក​បាន​រំលោភ​ពាក្យ​សន្យា​របស់​ខ្លួន​ដែល​បាន​ផ្តល់​សេចក្តី​ធានា​ថា នឹង​អនុញ្ញាត​ឱ្យ​នាវា​ស៊ីវិល​ធ្វើ​ចរាចរណ៍​ឆ្លង​កាត់​ច្រក​សមុទ្រ​អកមូហ្ស​ឡើង​វិញ»។ អាមេរិក​បាន​បន្ត​បិទ​ខ្ទប់​កំពង់ផែ​អ៊ីរ៉ង់​ដដែល។ អាមេរិក​បាន​ប្រកាស​កាលពី​ថ្ងៃ​ពុធ​ថា ខ្លួន​នឹង​បន្ត​បិទ​ខ្ទប់​កំពង់ផែ​អ៊ីរ៉ង់​រហូត​ដល់​ពេល​ណា​ដែល​អ៊ីរ៉ង់​មិន​គោរព​កិច្ចព្រមព្រៀង​ជាមួយ​សហរដ្ឋ​អាមេរិក។ អាមេរិក​បាន​បញ្ជាក់


In [ ]:
print("ARTICLE3", clean_gemma_output(summarize_long_khmer(ARTICLE3)))

ARTICLE3 ឆ្នាំ​២០១៥​មក​នេះ​ យោធា​អ៊ីស្រាអែល​បាន​ចាប់​ផ្តើម​ធ្វើ​ការ​ពង្រឹង​ការ​គ្រប់គ្រង​លើ​តំបន់​ភាគ​ខាង​ត្បូង​នៃ​ប្រទេស​លី​បង់​ តាម​ការ​អំពាវនាវ​របស់​សហរដ្ឋ​អាមេរិក​។ យោធា​អ៊ីស្រាអែល​បាន​ប្រកាស​ថា ខ្លួន​នឹង​អនុវត្ត​ការ​ពង្រឹង​ការ​គ្រប់គ្រង​នេះ​ តាម​ការ​អំពាវនាវ​របស់​សហរដ្ឋ​អាមេរិក​ តាម​ការ​អំពាវនាវ​របស់​សហភាព​អឺរ៉ុប​ និង​តាម​ការ​អំពាវនាវ​របស់​សហគមន៍​អន្តរជាតិ​ផ្សេងទៀត។ យោធា​អ៊ីស្រាអែល​បាន​និយាយ​ថា ខ្លួន​នឹង​អនុវត្ត​ការ​ពង្រឹង​ការ​គ្រប់គ្រង​នេះ​ តាម​ការ​អំពាវ​វើ​ង​របស់​សហរដ្ឋ​អាមេរិក​ តាម​ការ​អំពាវ​វើ​ង​របស់​សហភាព​អឺរ៉ុប​ និង​តាម​ការ​អំពាវ​វើ​ង​របស់​សហគមន៍​អន្តរជាតិ​ផ្សេង​ទៀត។ យោធា​អ៊ីស្រាអែល​បាន​និយាយ​ថា ខ្លួន​នឹង​អនុវត្ត​ការ​ពង្រឹង​ការ​គ្រប់គ្រង​នេះ​ តាម​ការ​អំពាវ​វើ​ង​របស់​សហរដ្ឋ​អាមេរិក​ តាម​ការ​អំពាវ​វើ​ង​របស់​សហភាព​អឺរ៉ុប​ និង​តាម​ការ​អំពាវ​វើ​ង​របស់​សហគមន៍​អន្តរជាតិ​ផ្សេង​ទៀត។ យោធា​អ៊ីស្រាអែល​បាន​និយាយ​ថា ខ្លួន​នឹង​អនុវត្ត​ការ​ពង្រឹង​ការ​គ្រប់គ្រង​នេះ​ តាម​ការ​អំពាវ​វើ​ង​របស់​សហរដ្ឋ


In [ ]:
print("ARTICLE4", clean_gemma_output(summarize_long_khmer(ARTICLE4)))

ARTICLE4 មាសដែលមានគុណភាពខ្ពស់ ព្រោះវាមានមាសសុទ្ធ 75%។ គ្រឿងអលង្ការដែលធ្វើពីមាស 18K អាចមានតម្លៃសមរម្យ ព្រមទាំងមានភាពរឹងមាំ ធន់ ងាយស្រួលក្នុងការថែទាំ និងអាចរក្សាបានយូរ។ គ្រឿងអលង្ការដែលធ្វើពីមាស 18K អាចមានតម្លៃខ្ពស់ជាងមាស 14K តែមានតម្លៃទាបជាងមាស 24K។

* មាសកម្រិត 14K៖

14K គឺជាមាសដែលត្រូវបានបង្កើតឡើងដោយផ្ទុកមាស 58.3% និងមានលោហៈធាតុ 41.7% ។ គ្រឿងអលង្ការដែលធ្វើពីមាស 14K អាចមានតម្លៃសមរម្យ ងាយស្រួលក្នុងការថែទាំ និងអាចរក្សាបានយូរ។ គ្រឿងអលង្ការដែលធ្វើពីមាស 14K អាចមានតម្លៃទាបជាងមាស 18K តែមានតម្លៃខ្ពស់ជាងមាស 10K។

* មាសកម្រិត 10K៖

10K គឺជាមាសដែលត្រូវបានបង្កើតឡើងដោយផ្ទុកមាស 41.7% និងមានលោហៈធាតុ 58.3% ។ គ្រឿងអលង្ការដែលធ្វើពីមាស 10K អាចមានតម្លៃទាប ងាយស្រួលក្នុងការថែទាំ និងអាចរក្សាបានយូរ។ គ្រឿងអលង្ការដែលធ្វើពីមាស 10K អាចមានតម្លៃទាបជាងមាស 14K តែមានតម្លៃទាបជាងមាស 18K និងមាស 24K។

មាសកម្រិត 10K, 14K, 18K និង 24K គឺជាកម្រិតនៃមាសសុទ្ធដែលត្រូវបានបញ្ចូលទៅក្នុងលោហៈធាតុផ្សេងទៀត ដើម្បីធ្វើជាគ្រឿងអលង្ការ។ គ្រឿងអលង្ការដែលធ្វើពីមាសកម្រិត 24K គឺជាមាសសុទ្ធ 100% តែមិនសូវមានភាពរឹងមាំ ងាយស្រួលឆ្កូត ងាយខូច ងាយបែក។ គ្រ